In [9]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

In [5]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

In [6]:
openai = OpenAI(api_key = OPENAI_KEY)
gemini = OpenAI(api_key = GEMINI_KEY, base_url = gemini_url)
ollama = OpenAI(base_url = ollama_url)

In [7]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [10]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

We have two volumes placed in order: 1st on the left, then 2nd, with their pages oriented in the same direction. Each volume has:

- Pages total thickness: 2 cm = 20 mm
- Each cover thickness: 2 mm

Structure from left to right for each volume:
Cover (front) thickness 2 mm, pages 20 mm, Cover (back) thickness 2 mm.

So the first volume spans from its left outside cover to its right outer cover:
Left cover 2 mm, pages 20 mm, right cover 2 mm → total 24 mm.

The second volume similarly spans 24 mm.

When the worm starts at the first page of the first volume (i.e., immediately after the left cover of the first volume) and ends at the last page of the second volume (i.e., immediately before the right cover of the second volume), it travels through the material in between. Since its path is perpendicular to pages, it goes straight through any intervening material along the straight line between those two pages.

What is directly in that line? It starts at the first page of volume 1 (just after its left cover). Then it must pass through:
- The remainder of the first volume's pages and the first volume’s right cover, until reaching the outer surface between volumes, then through the gap between volumes (assumed negligible since books are tightly side by side), then through the left cover of the second volume, and finally through the second volume’s pages up to its last page.

More simply: the worm travels from the first page of volume 1 to the last page of volume 2, so it must traverse:
- The remaining pages of volume 1 after the first page, plus the first cover’ thickness to the right edge, plus the thickness of the space between volumes (assumed zero), plus the left cover of volume 2, plus the pages of volume 2 up to the last page.

But note: "from the first page of the first volume" means starting at the very beginning of the pages of volume 1 (the first page), not after the first page. The starting point is at the leftmost page of volume 1, which is adjacent to the left cover. If we measure along the spine direction, the distance through material from the first page of vol 1 to the last page of vol 2 equals:

- Through the left cover of vol 1: 2 mm (but the worm starts at the first page, which is after the left cover, so it does not traverse the left cover of vol 1).
- Through the pages of vol 1 from first page to the end: 20 mm (the entire page stack).
- Through the right cover of vol 1: 2 mm.
- Through the space between volumes: 0 mm (assume they are pressed together).
- Through the left cover of vol 2: 2 mm.
- Through the pages of vol 2 from its first page to its last page: 20 mm.

Total = 20 + 2 + 2 + 20 = 44 mm.

Thus, the worm gnawed 44 millimeters (4.4 cm).

In [12]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

4.4 cm

Reason:
- Each volume has pages thickness 2 cm (20 mm) and two covers of 2 mm each (together 4 mm). So a volume is 24 mm thick.
- The worm starts at the first page of the first volume and ends at the last page of the second volume.
- From the first page of the first volume to the back cover of the first volume: it traverses the rest of the pages (about 20 mm) plus the back cover (2 mm) = about 22 mm.
- Between the books there is no gap to gnaw through (they’re side by side).
- From the front cover of the second volume to its last page: it traverses the front cover (2 mm) plus all the pages (20 mm) = 22 mm.
- Total gnawed distance = 22 mm + 22 mm = 44 mm = 4.4 cm.

In [13]:
response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

4 mm (0.4 cm).

Explanation: On a shelf with volumes I and II in order, the first page of volume I lies just inside its front cover, and the last page of volume II lies just inside its back cover. Those two covers face each other. So the worm only passes through the two covers: 2 mm + 2 mm = 4 mm.

In [14]:
response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 13.926551805s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{

In [18]:
response = ollama.chat.completions.create(model="gpt-oss:latest", messages=hard_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

The worm starts on the **first page** of the first volume (just inside the front cover of that book) and stops on the **last page** of the second volume (just outside the back cover of that book).  
The books are side‑by‑side, so the worm gnaws straight across the two volumes perpendicular to the pages.

The path the worm follows is the sum of the following thicknesses  

| segment | thickness |
|---------|-----------|
| pages of the first volume | \(2\ \text{cm}\) |
| back cover of the first volume | \(2\ \text{mm}=0.2\ \text{cm}\) |
| front cover of the second volume | \(2\ \text{mm}=0.2\ \text{cm}\) |
| pages of the second volume | \(2\ \text{cm}\) |

Adding them up:
\[
2\ \text{cm} + 0.2\ \text{cm} + 0.2\ \text{cm} + 2\ \text{cm}
= 4.4\ \text{cm}.
\]

\[
\boxed{4.4\ \text{cm}}
\]

In [17]:
response = ollama.chat.completions.create(model="gemma4:latest", messages=hard_puzzle, reasoning_effort="high")
display(Markdown(response.choices[0].message.content))

This is a problem of simple addition, but you must first ensure all units are consistent.

### 1. Standardize Units

We have measurements in centimeters (cm) and millimeters (mm). We will convert everything to **centimeters**.

*   $1 \text{ cm} = 10 \text{ mm}$
*   $2 \text{ mm} = 0.2 \text{ cm}$

### 2. Calculate the Total Thickness of ONE Volume ($T_{total}$)

Each volume consists of:
*   Pages: $2 \text{ cm}$
*   Front Cover: $2 \text{ mm} = 0.2 \text{ cm}$
*   Back Cover: $2 \text{ mm} = 0.2 \text{ cm}$

$$T_{\text{total}} = 2 \text{ cm} + 0.2 \text{ cm} + 0.2 \text{ cm} = 2.4 \text{ cm}$$

### 3. Calculate the Distance Gnawed

Since the two volumes are standing side by side, and the worm starts at the first page of the first volume and ends at the last page of the second volume, it passes through the entire material thickness of **both** volumes.

$$\text{Distance} = (\text{Thickness of Volume 1}) + (\text{Thickness of Volume 2})$$
$$\text{Distance} = 2.4 \text{ cm} + 2.4 \text{ cm}$$
$$\text{Distance} = 4.8 \text{ cm}$$

The worm gnawed through a distance of **4.8 cm**.

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke([HumanMessage(content=hard)])
display(Markdown(response.content))

To solve the problem, we need to calculate the total distance the worm gnawed through from the first page of the first volume to the last page of the second volume.

1. **Thickness of the Pages**: The thickness of each volume (including covers) is 2 cm.
   - Note: 1 cm = 10 mm, therefore 2 cm = 20 mm.

2. **Thickness of the Covers**: Each cover is 2 mm thick.

3. **Calculation of the thickness**:
   - The total thickness of the pages in each volume is the total thickness minus the thickness of the covers.
   - Each volume has covers of 2 mm on each side, so the thickness of the pages in one volume can be calculated as follows:
     \[
     \text{Thickness of pages} = \text{Total thickness} - 2 \times \text{Cover thickness}
     \]
     \[
     \text{Thickness of pages} = 20 \text{ mm} - 2 \times 2 \text{ mm} = 20 \text{ mm} - 4 \text{ mm} = 16 \text{ mm}
     \]
   - Therefore, the thickness of pages in each volume is 16 mm.

4. **Distance the worm gnawed**:
   - The worm starts at the first page of the first volume and goes through:
     - The thickness of the first volume’s pages (16 mm).
     - The thickness of the cover of the first volume (2 mm).
     - The thickness of the second volume (the pages) (16 mm).
     - The thickness of the cover of the second volume (2 mm).
     
   - Putting this all together, the total distance the worm gnawed is:
     \[
     \text{Total distance} = \text{Thickness of pages of the 1st volume} + \text{Cover thickness of the 1st volume} + \text{Thickness of pages of the 2nd volume} + \text{Cover thickness of the 2nd volume}
     \]
     \[
     \text{Total distance} = 16 \text{ mm} + 2 \text{ mm} + 16 \text{ mm} + 2 \text{ mm} = 36 \text{ mm}
     \]

5. **Final Output**:
   - Therefore, the total distance the worm gnawed through is **36 mm**.

In [47]:
from litellm import completion
from IPython.display import Markdown, display

tell_a_joke = "Tell me a joke about Hindu religion."

response = completion(
    model="openai/gpt-4.1",
    messages=[{"role": "user", "content": tell_a_joke}],
)

reply = response.choices[0].message.content
display(Markdown(reply))

Of course! It's important to approach topics related to religion with respect. Here's a lighthearted, respectful joke inspired by Hindu mythology:

**Why did Ganesha refuse to eat the computer?**

Because he didn’t want *bytes*, he prefers *laddus*!

Let me know if you'd like another one!

In [48]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 15
Output tokens: 64
Total tokens: 79
Total cost: 0.0542 cents


In [49]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Shakespeare's *Hamlet*, when Laertes returns to Denmark and demands to know "Where is my father?", the reply comes from **Claudius**.

Claudius, the King, tells him:

"**Come, go with me. I will go seek the king.**"

This is a bit of a trick answer from Claudius. He knows Hamlet has killed Laertes' father, Polonius, and he is trying to control the situation and manipulate Laertes. He doesn't directly answer "He's dead" or "Hamlet killed him." Instead, he feigns ignorance and offers to "seek the king" (implying he himself is the king and will lead Laertes to answers, all while steering him towards revenge against Hamlet).

In [50]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")
print(f"Total tokens: {response.usage.total_tokens}")

Input tokens: 19
Output tokens: 156
Total cost: 0.0064 cents
Total tokens: 175


In [56]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Shakespeare's *Hamlet*, when Laertes arrives at court in a fury and demands, "Where is my father?", the reply comes from **Claudius**.

Claudius tells him:

> "He is not dead, my lord."

This is a deceptive answer, as Claudius knows that Polonius is, in fact, dead, having been killed by Hamlet. Claudius is trying to placate Laertes and prevent him from immediately seeking revenge, as he intends to use Laertes's anger for his own purposes.

In [57]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output tokens: 110
Cached tokens: None
Total cost: 0.0046 cents
